# Training & Deploying An ARIMA Model Using Azure Machine Learning SDK

In [1]:
import warnings
warnings.filterwarnings('ignore')
from azureml.core import Workspace, Experiment, ScriptRunConfig, Dataset
from azureml.core.compute import AmlCompute, ComputeTarget
from azureml.core.environment import Environment
from azureml.core.model import Model, InferenceConfig
from azureml.core.webservice import AciWebservice
from sklearn.preprocessing import MinMaxScaler

## Configuring The Workspace

In [2]:
ws = Workspace.from_config()
print(ws.name, ws.resource_group, ws.location, ws.subscription_id, sep = '\n')

auotml-example-workspace
edwin.spartan117-rg
southeastasia
0c19fc19-85fd-4aa4-b133-61dd20fa93df


## Creating An Experiment

In [3]:
experiment_name = 'Energy-Demand-Forecasting'
experiment = Experiment(workspace = ws, name = experiment_name)

## Creating/Attaching A Compute Cluster

In [4]:
compute_name = 'CPU-Cluster'

if compute_name in ws.compute_targets:
    compute_target = ws.compute_targets[compute_name]
    if compute_target and type(compute_target) == AmlCompute:
        print('{} already exists. Just use the compute target'.format(compute_name))
else:
    print('Creating a new compute target...')
    provision_config = AmlCompute.provisioning_configuration(vm_size = 'Standard_A4m_v2', min_nodes = 0, max_nodes = 4)
    compute_target = ComputeTarget.create(ws, compute_name, provision_config)
    compute_target.wait_for_completion(show_output = True, min_node_count = 0, timeout_in_minutes = 20)
    
# For a more detailed view of the current AmlCompute status, use the get_status() method
compute_target.get_status().serialize()

CPU-Cluster already exists. Just use the compute target


{'currentNodeCount': 1,
 'targetNodeCount': 1,
 'nodeStateCounts': {'preparingNodeCount': 0,
  'runningNodeCount': 0,
  'idleNodeCount': 1,
  'unusableNodeCount': 0,
  'leavingNodeCount': 0,
  'preemptedNodeCount': 0},
 'allocationState': 'Steady',
 'allocationStateTransitionTime': '2025-02-18T04:12:40.684000+00:00',
 'errors': None,
 'creationTime': '2025-02-17T16:48:50.205132+00:00',
 'modifiedTime': '2025-02-17T16:50:17.526243+00:00',
 'provisioningState': 'Succeeded',
 'provisioningStateTransitionTime': None,
 'scaleSettings': {'minNodeCount': 0,
  'maxNodeCount': 4,
  'nodeIdleTimeBeforeScaleDown': 'PT900S'},
 'vmPriority': 'Dedicated',
 'vmSize': 'Standard_A4m_v2'}

## Uploading Data To A Datastore

In [5]:
ds = ws.get_default_datastore()
print(ds.name, ds.datastore_type, ds.account_name, ds.container_name, sep = '\n')

workspaceblobstore
AzureBlob
auotmlexamplew5880114168
azureml-blobstore-080e5758-0b91-49eb-ac97-661db0f2b2eb


In [6]:
ds.upload(src_dir = '../Data', target_path = 'GEFCom2014 Energy Load Forecasting', overwrite = True, show_progress = True)

"Datastore.upload" is deprecated after version 1.0.69. Please use "Dataset.File.upload_directory" to upload your files             from a local directory and create FileDataset in single method call. See Dataset API change notice at https://aka.ms/dataset-deprecation.


Uploading an estimated of 12 files
Uploading ../Data/.amlignore
Uploaded ../Data/.amlignore, 1 files out of an estimated total of 12
Uploading ../Data/.amlignore.amltmp
Uploaded ../Data/.amlignore.amltmp, 2 files out of an estimated total of 12
Uploading ../Data/.DS_Store
Uploaded ../Data/.DS_Store, 3 files out of an estimated total of 12
Uploading ../Data/Energy.csv
Uploaded ../Data/Energy.csv, 4 files out of an estimated total of 12
Uploading ../Data/GEFCom2014-E.xlsx
Uploaded ../Data/GEFCom2014-E.xlsx, 5 files out of an estimated total of 12
Uploading ../Data/GEFCom2014-L_V2.zip
Uploaded ../Data/GEFCom2014-L_V2.zip, 6 files out of an estimated total of 12
Uploading ../Data/GEFCom2014-P_V2.zip
Uploaded ../Data/GEFCom2014-P_V2.zip, 7 files out of an estimated total of 12
Uploading ../Data/Provisional_Leaderboard_V2.xlsx
Uploaded ../Data/Provisional_Leaderboard_V2.xlsx, 8 files out of an estimated total of 12
Uploading ../Data/READ ME_V2.txt
Uploaded ../Data/READ ME_V2.txt, 9 files out

$AZUREML_DATAREFERENCE_51a12740d64d4df08270571ff5b1aa3e

## Getting (Creating & Registering) An Environment

In [7]:
# ts_forecasting_env = Environment.from_conda_specification(name = 'Time Series Forecasting', file_path = 'Energy Demand Forecasting/azureml-env.yml')
# ts_forecasting_env.register(workspace = ws)

ts_forecasting_env = Environment.get(workspace = ws, name = 'Time Series Forecasting')

## Training & Registering The SARIMAX Model

### Submitting The Training Job To A Compute Cluster

In [8]:
script_args = ['--data-folder', Dataset.File.from_files(path = (ds, 'GEFCom2014 Energy Load Forecasting')).as_mount(), 
               '--filename', 'Energy.csv']

script_config = ScriptRunConfig(source_directory = 'Energy Demand Forecasting',  script = 'train.py', 
                                arguments = script_args, compute_target = compute_target, environment = ts_forecasting_env)

In [9]:
experiment_run = experiment.submit(config = script_config)
experiment_run.wait_for_completion(show_output = True)

RunId: Energy-Demand-Forecasting_1739853041_b8aaf7fd
Web View: https://ml.azure.com/runs/Energy-Demand-Forecasting_1739853041_b8aaf7fd?wsid=/subscriptions/0c19fc19-85fd-4aa4-b133-61dd20fa93df/resourcegroups/edwin.spartan117-rg/workspaces/auotml-example-workspace&tid=c5f4b1c2-b533-4788-b1c5-99d0f10fb9b6

Streaming user_logs/std_log.txt

/azureml-envs/azureml_c687b23f5a6f938fe73735ca252da2cb/lib/python3.6/site-packages/jwt/utils.py:8: CryptographyDeprecationWarning: Python 3.6 is no longer supported by the Python core team. Therefore, support for it is deprecated in cryptography. The next release of cryptography will remove support for Python 3.6.
  from cryptography.hazmat.primitives.asymmetric.utils import (
Cleaning up all outstanding Run operations, waiting 300.0 seconds
0 items cleaning up...
Cleanup took 1.9073486328125e-06 seconds

Execution Summary
RunId: Energy-Demand-Forecasting_1739853041_b8aaf7fd
Web View: https://ml.azure.com/runs/Energy-Demand-Forecasting_1739853041_b8aaf7f

{'runId': 'Energy-Demand-Forecasting_1739853041_b8aaf7fd',
 'target': 'CPU-Cluster',
 'status': 'Completed',
 'startTimeUtc': '2025-02-18T04:31:14.691721Z',
 'endTimeUtc': '2025-02-18T04:33:05.43149Z',
 'services': {},
 'properties': {'_azureml.ComputeTargetType': 'amlctrain',
  '_azureml.ClusterName': 'CPU-Cluster',
  'ContentSnapshotId': '69b5b730-2051-4edb-864e-bce0dc29ee42',
  'ProcessInfoFile': 'azureml-logs/process_info.json',
  'ProcessStatusFile': 'azureml-logs/process_status.json'},
 'inputDatasets': [{'dataset': {'id': '264c4e4e-9fc1-49c3-aba3-be6d47135fab'}, 'consumptionDetails': {'type': 'RunInput', 'inputName': 'input__4c3ee259', 'mechanism': 'Mount'}}],
 'outputDatasets': [],
 'runDefinition': {'script': 'train.py',
  'command': '',
  'useAbsolutePath': False,
  'arguments': ['--data-folder',
   'DatasetConsumptionConfig:input__4c3ee259',
   '--filename',
   'Energy.csv'],
  'sourceDirectoryDataStore': None,
  'framework': 'Python',
  'communicator': 'None',
  'target': '

In [10]:
experiment_run.get_file_names()

['logs/azureml/dataprep/python_span_883ad6ef-a851-4291-9ec4-8c3f7effe552.jsonl',
 'outputs/SARIMAX Energy Forecasting.pkl',
 'system_logs/cs_capability/cs-capability.log',
 'system_logs/data_capability/data-capability.log',
 'system_logs/data_capability/rslex.log.2025-02-18-04',
 'system_logs/hosttools_capability/hosttools-capability.log',
 'system_logs/lifecycler/execution-wrapper.log',
 'system_logs/lifecycler/lifecycler.log',
 'system_logs/metrics_capability/metrics-capability.log',
 'system_logs/snapshot_capability/snapshot-capability.log',
 'user_logs/std_log.txt']

In [11]:
model = experiment_run.register_model(model_name = 'SARIMAX-Energy-Forecasting', model_path = 'outputs/SARIMAX Energy Forecasting.pkl')
# model.download(target_dir = 'Energy Demand Forecasting', exist_ok = True)

## Model Deployment & Evaluation

In [ ]:
inference_config = InferenceConfig(source_directory = 'Energy Demand Forecasting', entry_script = 'score.py',
                                   environment = ts_forecasting_env)
deployment_config = AciWebservice.deploy_configuration(cpu_cores = 2, memory_gb = 8)
web_service = Model.deploy(workspace = ws, name = 'arima-aci-service', models = [model], 
                           inference_config = inference_config, deployment_config = deployment_config)
web_service.wait_for_deployment(show_output = True)

In [14]:
import sys
sys.path.append('Energy Demand Forecasting')
import numpy as np
import joblib
from helper_functions import load_data, MAPE

energy_forecasting_df = load_data('../Data/Energy.csv')
prediction_df = energy_forecasting_df.iloc[2185:2190]

scaler = MinMaxScaler()
energy_forecasting_df['Load'] = scaler.fit_transform(np.array(energy_forecasting_df.loc[:, 'Load'].values).reshape(-1, 1))

model.download(target_dir = './', exist_ok = True)
energy_forecasting_model = joblib.load('SARIMAX Energy Forecasting.pkl')
prediction = energy_forecasting_model.forecast(steps = 5)
prediction_df['Prediction'] = prediction

prediction_df.columns = ['Actual', 'Prediction']
prediction_df[['Actual', 'Prediction']] = scaler.inverse_transform(prediction_df[['Actual', 'Prediction']])

energy_forecasting_MAPE = MAPE(prediction_df['Actual'], prediction_df['Prediction'])
print('MAPE = {:.4f}%, Accuracy = {:.2f}%'.format(energy_forecasting_MAPE * 100, (1 - energy_forecasting_MAPE) * 100))

MAPE = 4.5596%, Accuracy = 95.44%
